# PubChem RI-windowed retrieval (StdNP) — publication figure

Top-1 (and top-k) accuracy vs. candidate-set size (RI window: 1k/100k/1M/10M),
plus the true full-test-set "all" (no RI restriction) as the rightmost point,
for ICICLE, NEIMS, and MassFormer, both `inject` and `autofail` modes.

Reads two files per model and merges them:
- `retrieval_ablation_StdNP.json` — the StdNP RI-window ladder (1000/100000/
  1000000/10000000 levels), restricted to test molecules that have an RI_StdNP
  value (a small subset — ~1,259 for ICICLE/NEIMS, out of ~33.5k test
  molecules total).
- `retrieval_global_results.json` — the true global "all" scan (no RI
  restriction), scored against the FULL test set (~27.6k molecules with a
  valid GT spectrum). This is a different, larger query set than the
  StdNP-restricted ladder — see the note in the load cell.

ICICLE's predictions here are from the corrected re-run
(`pubchem_predictions_rerun_260710.hdf5`) — the checkpoint/resume race,
zero-edge-graph crash, and batch-boundary misalignment bugs that corrupted
the original PubChem-scale predictions are all fixed and independently
verified (MW-violation rate 0.004% on the full 93.5M-row output, matching
NEIMS's clean baseline once accounting for expected halogen/Se isotope
patterns).

SemiStdNP and StdPolar RI types, plus the MW-only (±80Da) and RI∪MW-union
tracks, are still running as of this writing — this notebook covers StdNP
only for now and can be extended once those land.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from icicle.utils.visualization.eval_plots import plot_ri_ladder
from icicle.utils.visualization.style import save_fig, set_style

set_style("manuscript")

RESULTS = Path("/home/magled/icicle-dev/results")
OUTPUT_DIR = Path("figures/retrieval_ri")
MODEL_DIRS = {
    "ICICLE": RESULTS / "pubchem_retrieval_eval_icicle_rerun_260710",
    "NEIMS": RESULTS / "pubchem_retrieval_eval_neims",
    "MassFormer": RESULTS / "pubchem_retrieval_eval_massformer",
}
RI_TYPE = "StdNP"
METRIC = "cosine"
K_VALUES = [1, 5, 10, 50]

## Load

In [ ]:
model_results = {}
for label, d in MODEL_DIRS.items():
    ladder_path = d / f"retrieval_ablation_{RI_TYPE}.json"
    global_path = d / "retrieval_global_results.json"

    results = {}
    if ladder_path.exists():
        with open(ladder_path) as f:
            ladder = json.load(f)
        # Ladder levels are keyed as plain strings ("1000", "100000", ...)
        # plus its own RI-restricted "all" level -- keep only the numeric
        # windowed levels here; the "all" point below comes from the
        # separate full-test-set run instead (different, larger query set).
        results.update({k: v for k, v in ladder.items() if k != "all"})
    else:
        print(f"{label}: {ladder_path} not found")

    if global_path.exists():
        with open(global_path) as f:
            global_results = json.load(f)
        # retrieval_global_results.json's top-level key is "all" already.
        results.update(global_results)
    else:
        print(f"{label}: {global_path} not found")

    if results:
        model_results[label] = results
        print(f"{label}: levels = {list(results.keys())}")

print(
    "\nNote: the windowed levels (1000/100000/1000000/10000000) are scored "
    "against only the test molecules with an RI_StdNP value (a small "
    "subset), while the 'all' point is scored against the FULL test set "
    "with a valid GT spectrum (~27.6k molecules) -- these are two "
    "different query sets, not a single continuously-growing one. Treat "
    "the 'all' point as a separate reference/ceiling, not a strict "
    "continuation of the ladder."
)

## Top-k accuracy vs. candidate-set size

One figure per (top-k × mode). Five x-values per model: 1000, 100000,
1000000, 10000000 (StdNP-restricted query set), and "all" (full test set,
separate query set — see the caveat above).

In [ ]:
for mode in ["inject", "autofail"]:
    for k in K_VALUES:
        fig = plot_ri_ladder(model_results, metric=METRIC, mode=mode, k=k)
        fig.axes[0].set_title(f"{mode}")
        save_fig(fig, f"retrieval_ri_top{k}_{mode}", OUTPUT_DIR)
        plt.show()
        plt.close(fig)

## Summary table

In [ ]:
rows = []
for label, results in model_results.items():
    for level, by_mode in results.items():
        for mode, by_metric in by_mode.items():
            m = by_metric.get(METRIC)
            if not m:
                continue
            row = {"model": label, "candidate_set_size": level, "mode": mode}
            for k in K_VALUES:
                row[f"top-{k}"] = f"{100 * m[f'top_{k}_accuracy']:.1f}"
            row["mrr"] = f"{m['mrr']:.4f}"
            row["median_rank"] = m["median_rank"]
            rows.append(row)

df_summary = pd.DataFrame(rows)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(OUTPUT_DIR / "retrieval_ri_summary.csv", index=False)
df_summary